# 06 - Inference and API Simulation

**Objectif :** charger le bundle, scorer des dossiers bruts, puis vérifier le comportement API in-process.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Load external holdout

In [ ]:
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

holdout_df = CsvLoanDataLoader(path=settings.raw_test_path).load()
holdout_df.head()

## 2. Load model bundle

In [ ]:
from credit_risk_lab.infrastructure.modeling import JoblibModelBundleRepository

repository = JoblibModelBundleRepository()
bundle = repository.load(settings.model_bundle_path)
bundle["metadata"]

## 3. Score raw applications

In [ ]:
from credit_risk_lab.application import RawLoanScorer

scorer = RawLoanScorer(bundle, threshold=settings.decision_threshold)
scoring_result = scorer.score(holdout_df.head(20))

pd.DataFrame(
    {
        "probability_of_risk": scoring_result.probabilities,
        "risk_decision": scoring_result.decisions,
    }
).head()

## 4. Validate one API payload

In [ ]:
from credit_risk_lab.interfaces.api_models import LoanApplication

payload = holdout_df.drop(columns=[settings.target_column]).iloc[0].to_dict()
application = LoanApplication.model_validate(payload)
application

## 5. Predict through API service

In [ ]:
from credit_risk_lab.interfaces.api_service import predict_application

response = predict_application(application, request_id="notebook-demo")
response

## 6. In-process API simulation

In [ ]:
from credit_risk_lab.interfaces.api_simulation import run_api_simulation

simulation = run_api_simulation(limit=20)
display(simulation.health)
display(simulation.responses.head())
{"invalid_status_code": simulation.invalid_status_code}

## 7. Simulation distribution

In [ ]:
px.histogram(
    simulation.responses,
    x="probability_of_risk",
    nbins=20,
    title="Simulated API risk probabilities",
    template="plotly_white",
).show()